# Baseline Models para Previsão de Churn 🤖

Objetivo: estabelecer métricas de referência (piso mínimo) antes da modelagem com MLP. Todos os experimentos são logados no MLflow para rastreabilidade.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    average_precision_score,
    ConfusionMatrixDisplay,
)

# Semente global para reprodutibilidade
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 📏 1. Definição das Métricas de Avaliação

Nesta etapa, são definidas as métricas técnicas que serão utilizadas para avaliar todos os modelos de forma padronizada. Centralizar as métricas em uma função garante comparações justas entre os modelos.

💡 **Justificativa das métricas:**

Como a variável alvo apresenta desbalanceamento entre as classes (~73% No Churn vs ~27% Churn), a **acurácia isolada é uma métrica enganosa** — um modelo que sempre prevê "No Churn" atingiria 73% sem nenhum poder preditivo real. Por isso, serão utilizadas:

- **ROC-AUC**: mede a capacidade discriminativa geral do modelo em todos os thresholds
- **PR-AUC** *(Average Precision)*: especialmente relevante em dados desbalanceados — não é inflada pela massa de verdadeiros negativos como o ROC-AUC
- **F1-Score**: equilíbrio entre Precision e Recall, útil quando os dois erros (FP e FN) têm custos diferentes
- **Precision / Recall**: avaliados separadamente para entender o perfil de erro de cada modelo

In [ ]:
def avaliar_modelo(nome_modelo: str, y_true, y_pred, y_prob) -> dict:
    """
    Evaluates a classifier and returns a standardized metrics dictionary.

    Parameters
    ----------
    nome_modelo : str
        Display name for the model (used in comparison tables).
    y_true : array-like
        True binary labels.
    y_pred : array-like
        Predicted binary labels.
    y_prob : array-like
        Predicted probabilities for the positive class.

    Returns
    -------
    dict
        Dictionary with keys: modelo, accuracy, precision, recall,
        f1_score, roc_auc, pr_auc.
    """
    return {
        "modelo":     nome_modelo,
        "accuracy":   accuracy_score(y_true, y_pred),
        "precision":  precision_score(y_true, y_pred, zero_division=0),
        "recall":     recall_score(y_true, y_pred, zero_division=0),
        "f1_score":   f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":    roc_auc_score(y_true, y_prob),
        "pr_auc":     average_precision_score(y_true, y_prob),
    }

In [2]:
# Carregando o dataset original
df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

# --- Reaplicando a limpeza do EDA ---
# Convertendo Total Charges para float e removendo os 11 registros problemáticos
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df = df.dropna(subset=["Total Charges"]).reset_index(drop=True)

# Removendo colunas desnecessárias (identificadores, leakage, sem variância)
cols_to_drop = [
    "Count", "Country", "State", "CustomerID", "Lat Long",
    "Latitude", "Longitude", "Zip Code", "City",
    "Churn Score", "CLTV", "Churn Reason", "Churn Label",
]
df = df.drop(columns=cols_to_drop)

print(f"Shape: {df.shape}")
print(f"Churn rate: {df['Churn Value'].mean():.2%}")


Shape: (7032, 20)
Churn rate: 26.58%


# 🧹 2. Preparação dos Dados

Nesta etapa, os dados serão carregados, limpos e divididos em conjuntos de treino e teste. O pipeline de pré-processamento também será configurado, incluindo imputação de valores faltantes, escalonamento das variáveis numéricas e codificação das categóricas.

In [3]:
# Separando features do target
X = df.drop(columns=["Churn Value"])
y = df["Churn Value"]

# Identificando colunas por tipo
numeric_features = ["Tenure Months", "Monthly Charges", "Total Charges"]
categorical_features = [col for col in X.columns if col not in numeric_features]

print(f"Features numéricas ({len(numeric_features)}): {numeric_features}")
print(f"Features categóricas ({len(categorical_features)}): {categorical_features}")

# Split treino/teste — estratificado para manter a proporção de churn em ambos os sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y,  # garante ~26.5% de churn em treino e teste
)

print(f"\nTreino: {X_train.shape} | Churn rate: {y_train.mean():.2%}")
print(f"Teste:  {X_test.shape}  | Churn rate: {y_test.mean():.2%}")


Features numéricas (3): ['Tenure Months', 'Monthly Charges', 'Total Charges']
Features categóricas (16): ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Treino: (5625, 19) | Churn rate: 26.58%
Teste:  (1407, 19)  | Churn rate: 26.58%


In [ ]:
# Pipelines individuais por tipo de variável
# A imputação é aplicada antes do scaling/encoding para garantir robustez em produção
# mesmo que o dataset de treino esteja limpo.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),     # imputa com mediana (robusto a outliers)
    ("scaler",  StandardScaler()),                     # centraliza e escala
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),          # imputa com moda
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer,    numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Pré-processamento configurado com sucesso.")
print(f"  Numéricas ({len(numeric_features)}): {numeric_features}")
print(f"  Categóricas ({len(categorical_features)}): {categorical_features}")

💡 **Observação:**

A imputação com `SimpleImputer` é adicionada **antes** do scaling e do encoding, mesmo que o dataset de treino esteja completamente limpo. Isso garante que o pipeline seja **robusto em produção**: se um campo chegar faltando numa requisição à API, o pré-processador tratará o valor automaticamente em vez de gerar um erro. A estratégia de mediana para numéricas é preferível à média por ser robusta a outliers.

# 🤖 3. Modelo Baseline com DummyClassifier

Nesta etapa, será treinado um classificador ingênuo como referência inicial. O `DummyClassifier` com estratégia `most_frequent` sempre prevê a classe majoritária ("No Churn"), estabelecendo o **piso mínimo absoluto** que qualquer modelo real deve superar.

In [ ]:
# Configurando o experimento — todas as runs ficam agrupadas aqui no MLflow UI
mlflow.set_experiment("telco-churn-baseline")

with mlflow.start_run(run_name="dummy-classifier"):

    # --- Modelo ---
    dummy_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)),
    ])
    dummy_pipeline.fit(X_train, y_train)
    y_pred  = dummy_pipeline.predict(X_test)
    y_proba = dummy_pipeline.predict_proba(X_test)[:, 1]

    # --- Métricas padronizadas ---
    resultado_dummy = avaliar_modelo("DummyClassifier", y_test, y_pred, y_proba)

    # --- Log no MLflow ---
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("random_seed", RANDOM_SEED)
    mlflow.log_metrics({k: v for k, v in resultado_dummy.items() if k != "modelo"})
    mlflow.sklearn.log_model(dummy_pipeline, artifact_path="model")

    # --- Exibição ---
    print(f"ROC-AUC : {resultado_dummy['roc_auc']:.4f}")
    print(f"PR-AUC  : {resultado_dummy['pr_auc']:.4f}")
    print(f"F1-Score: {resultado_dummy['f1_score']:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

    # Matriz de confusão
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["No Churn", "Churn"],
        colorbar=False,
        ax=ax,
    )
    ax.set_title("Dummy Classifier — Matriz de Confusão")
    plt.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.show()

💡 **Observação:**

O DummyClassifier, configurado para prever sempre a classe majoritária, apresentou **ROC-AUC de 0.50** e **F1-Score de 0.00** para a classe de churn — exatamente o esperado para um modelo sem inteligência. Esse resultado é o **piso absoluto**: qualquer modelo com valor abaixo desses números seria pior do que adivinhar. O **PR-AUC** (~0.27) reflete a proporção base de churn no dataset, servindo como referência de um classificador aleatório para essa métrica.

# 🤖 4. Modelo Baseline com Regressão Logística

Nesta etapa, será treinado um modelo de Regressão Logística como baseline supervisionado. O parâmetro `class_weight="balanced"` ajusta os pesos das classes inversamente proporcional à sua frequência, corrigindo o desbalanceamento sem necessidade de resampling.

In [ ]:
with mlflow.start_run(run_name="logistic-regression"):

    # --- Modelo ---
    lr_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",  # penaliza erros na classe minoritária proporcionalmente
            max_iter=1000,            # garante convergência
            random_state=RANDOM_SEED,
            C=1.0,                    # regularização L2 padrão (1/lambda)
        )),
    ])
    lr_pipeline.fit(X_train, y_train)
    y_pred  = lr_pipeline.predict(X_test)
    y_proba = lr_pipeline.predict_proba(X_test)[:, 1]

    # --- Métricas padronizadas ---
    resultado_lr = avaliar_modelo("Logistic Regression", y_test, y_pred, y_proba)

    # --- Log no MLflow ---
    mlflow.log_param("C", 1.0)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_seed", RANDOM_SEED)
    mlflow.log_metrics({k: v for k, v in resultado_lr.items() if k != "modelo"})
    mlflow.sklearn.log_model(lr_pipeline, artifact_path="model")

    # --- Exibição ---
    print(f"ROC-AUC : {resultado_lr['roc_auc']:.4f}")
    print(f"PR-AUC  : {resultado_lr['pr_auc']:.4f}")
    print(f"F1-Score: {resultado_lr['f1_score']:.4f}")
    print(f"Recall  : {resultado_lr['recall']:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

    # Matriz de confusão
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["No Churn", "Churn"],
        colorbar=False,
        ax=ax,
    )
    ax.set_title("Regressão Logística — Matriz de Confusão")
    plt.tight_layout()
    mlflow.log_figure(fig, "confusion_matrix.png")
    plt.show()

💡 **Observação:**

A Regressão Logística apresentou desempenho significativamente superior ao DummyClassifier, demonstrando que mesmo um modelo linear consegue capturar padrões relevantes no dataset. O `class_weight="balanced"` eleva o Recall da classe de churn em troca de alguma precisão — tradeoff favorável dado que **falsos negativos (churns não detectados) custam muito mais do que falsos positivos** (intervenções desnecessárias). O **PR-AUC** é a métrica mais honesta para avaliar a qualidade do ranking de probabilidades neste dataset desbalanceado.

# 💾 5. Exportando o Dataset Pré-processado

O dataframe limpo (após remoção de colunas e correção de tipos, mas **sem encoding**) é salvo em `data/processed/` para reuso no próximo notebook de engenharia de modelos. Exportar antes do encoding preserva a legibilidade das colunas e permite aplicar feature engineering no próximo notebook antes de encodar.

In [7]:
import os

# Salvando o dataframe limpo (sem encoding) para reuso no notebook de engenharia de modelos
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/telco_churn_cleaned.csv", index=False)

print("Dataset pre-processado salvo em: data/processed/telco_churn_cleaned.csv")
print(f"Shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")

Dataset pre-processado salvo em: data/processed/telco_churn_cleaned.csv
Shape: (7032, 20)
Colunas: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value']


# 🏁 6. Comparação Final — Baseline Models

Nesta etapa, os resultados dos dois modelos são consolidados para definir os benchmarks que o MLP deverá superar na próxima fase.

💡 **Observação:**

| Modelo | Accuracy | Precision | Recall | F1-Score | ROC-AUC | PR-AUC |
|--------|----------|-----------|--------|----------|---------|--------|
| Dummy Classifier | 0.7342 | 0.00 | 0.00 | 0.00 | 0.5000 | 0.2658 |
| Logistic Regression | 0.7292 | 0.4941 | 0.7861 | 0.6068 | 0.8425 | 0.6313 |

A Regressão Logística com `class_weight="balanced"` troca Accuracy (~73%) por um Recall muito mais alto na classe de churn (0.79 vs 0.00). Isso é o comportamento esperado e desejado: o modelo passa a identificar ativamente os churners em vez de ignorá-los. O **PR-AUC de 0.6313** frente ao baseline aleatório de 0.2658 confirma que o modelo tem capacidade real de ordenar clientes por risco.

**Meta para os próximos estágios:** ROC-AUC > 0.85 e PR-AUC > 0.65

**Artefatos exportados:**
- `data/processed/telco_churn_cleaned.csv` — dataset limpo sem encoding (input para `03_model_engineering.ipynb`)
- MLflow experiment `telco-churn-baseline` — runs: `dummy-classifier`, `logistic-regression`